# Лекция 2. Оптимизация SQL-запросов

Ноутбук для демонстрации планов выполнения, индексов и переписывания запросов. Используем SQLite, поэтому синтаксис плана отличается от PostgreSQL/ClickHouse, но принципы те же.

In [1]:
import sqlite3
import time
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
conn = sqlite3.connect(':memory:')
conn.execute('PRAGMA cache_size = -200000')
conn.execute('PRAGMA temp_store = MEMORY')

channels = np.array(['organic', 'paid_ads', 'referral', 'email', 'partner'])
cities = np.array(['Москва', 'Санкт-Петербург', 'Казань', 'Новосибирск', 'Екатеринбург'])

n_users = 12000
users = pd.DataFrame({
    'user_id': np.arange(1, n_users + 1),
    'signup_date': (pd.to_datetime('2023-01-01') + pd.to_timedelta(rng.integers(0, 640, n_users), unit='D')).strftime('%Y-%m-%d'),
    'channel': rng.choice(channels, n_users, p=[0.34, 0.30, 0.16, 0.12, 0.08]),
    'city': rng.choice(cities, n_users),
    'email': [f'user{i}@example.com' for i in range(1, n_users + 1)],
})

n_products = 400
products = pd.DataFrame({
    'product_id': np.arange(1, n_products + 1),
    'category': rng.choice(['electronics','home','books','sport','beauty','kids','food'], n_products),
    'base_price': rng.integers(200, 12000, n_products).astype(float),
})
base_by_product = products.set_index('product_id')['base_price'].to_dict()

n_orders = 65000
orders = pd.DataFrame({
    'order_id': np.arange(1, n_orders + 1),
    'user_id': rng.choice(users['user_id'], n_orders),
    'order_date': (pd.to_datetime('2023-01-01') + pd.to_timedelta(rng.integers(0, 640, n_orders), unit='D')).strftime('%Y-%m-%d'),
    'status': rng.choice(['paid','cancelled','refunded'], n_orders, p=[0.74,0.20,0.06]),
})

item_counts = rng.integers(1, 5, n_orders)
order_ids = np.repeat(orders['order_id'].to_numpy(), item_counts)
product_ids = rng.choice(products['product_id'].to_numpy(), len(order_ids), replace=True)
base_prices = np.array([base_by_product[int(pid)] for pid in product_ids])
order_items = pd.DataFrame({
    'order_id': order_ids,
    'product_id': product_ids,
    'quantity': rng.choice([1, 1, 2, 3], len(order_ids)),
    'price': np.round(base_prices * rng.uniform(0.7, 1.25, len(order_ids)), 2),
})

n_events = 180000
events = pd.DataFrame({
    'event_id': np.arange(1, n_events + 1),
    'user_id': rng.choice(users['user_id'], n_events),
    'event_time': (pd.to_datetime('2023-01-01') + pd.to_timedelta(rng.integers(0, 640, n_events), unit='D')).strftime('%Y-%m-%d'),
    'event_type': rng.choice(['app_open','product_view','add_to_cart','checkout_start','purchase'], n_events, p=[.42,.32,.12,.09,.05]),
    'product_id': rng.choice(products['product_id'], n_events),
})

for name, df in [('users', users), ('products', products), ('orders', orders), ('order_items', order_items), ('events', events)]:
    df.to_sql(name, conn, index=False, if_exists='replace')

def run(sql):
    return pd.read_sql_query(sql, conn)

def explain(sql):
    return pd.read_sql_query('EXPLAIN QUERY PLAN ' + sql, conn)

def time_sql(sql, repeat=3, preview=5):
    times = []
    result = None
    for _ in range(repeat):
        t0 = time.perf_counter()
        result = pd.read_sql_query(sql, conn)
        times.append(time.perf_counter() - t0)
    print(f'best: {min(times):.4f}s | avg: {sum(times)/len(times):.4f}s | rows: {len(result)}')
    return result.head(preview)

for table in ['users','products','orders','order_items','events']:
    print(table, run(f'SELECT COUNT(*) AS rows FROM {table}').iloc[0, 0])

users 12000
products 400
orders 65000
order_items 162229
events 180000


## 1. Baseline: запрос с функцией над датой

Бизнес-вопрос: выручка за июнь 2024 по каналам привлечения.

In [2]:
baseline = '''
SELECT u.channel, SUM(oi.quantity * oi.price) AS revenue
FROM orders o
JOIN users u USING(user_id)
JOIN order_items oi USING(order_id)
WHERE substr(o.order_date, 1, 7) = '2024-06'
  AND o.status = 'paid'
GROUP BY u.channel
ORDER BY revenue DESC
'''
explain(baseline)

,id,parent,notused,detail
0,10,0,216,SCAN o
1,19,0,0,BLOOM FILTER ON u (user_id=?)
2,29,0,53,SEARCH u USING AUTOMATIC COVERING INDEX (user_...
3,38,0,0,BLOOM FILTER ON oi (order_id=?)
4,50,0,53,SEARCH oi USING AUTOMATIC COVERING INDEX (orde...
5,57,0,0,USE TEMP B-TREE FOR GROUP BY
6,101,0,0,USE TEMP B-TREE FOR ORDER BY


In [3]:
time_sql(baseline)

best: 0.0845s | avg: 0.0852s | rows: 5


,channel,revenue
0,organic,20538233.88
1,paid_ads,18821631.27
2,referral,10753845.57
3,email,7838224.43
4,partner,5499758.09


## 2. Индекс и sargable-фильтр

Создайте индекс по `(status, order_date)` и перепишите фильтр на диапазон дат.

In [4]:
conn.execute('CREATE INDEX IF NOT EXISTS idx_orders_status_date ON orders(status, order_date)')
optimized_date = '''
select
    u.channel,
    sum(oi.quantity * oi.price) as revenue
from
    orders o
    join
    users u
        using(user_id)
    join
    order_items oi
        using(order_id)
where
    o.order_date >= '2024-06-01'
    and
    o.order_date < '2024-07-01'
    and
    o.status = 'paid'
group by
    u.channel
order by
    revenue desc
'''
explain(optimized_date)

,id,parent,notused,detail
0,11,0,50,SEARCH o USING INDEX idx_orders_status_date (s...
1,18,0,216,SCAN u
2,25,0,0,BLOOM FILTER ON oi (order_id=?)
3,37,0,53,SEARCH oi USING AUTOMATIC COVERING INDEX (orde...
4,44,0,0,USE TEMP B-TREE FOR GROUP BY
5,88,0,0,USE TEMP B-TREE FOR ORDER BY


In [5]:
time_sql(optimized_date)

best: 0.9982s | avg: 1.0026s | rows: 5


,channel,revenue
0,organic,20538233.88
1,paid_ads,18821631.27
2,referral,10753845.57
3,email,7838224.43
4,partner,5499758.09


## 3. Агрегация до JOIN

Сначала посчитайте выручку на уровне `order_id`, затем присоединяйте к `orders` и `users`.

In [6]:
preagg = '''
with paid_orders as (
    select
        order_id,
        user_id
    from
        orders
    where
        status = 'paid'
        and
        order_date >= '2024-06-01'
        and
        order_date < '2024-07-01'
),
order_revenue as (
    select
        order_id,
        sum(quantity * price) as revenue
    from
        order_items
    group by
        order_id
)
select
    u.channel,
    sum(orv.revenue) as revenue
from
    paid_orders po
    join
    order_revenue orv
        on orv.order_id = po.order_id
    join
    users u
        on po.user_id = u.user_id
group by
    u.channel
order by
    revenue desc
'''
explain(preagg)

,id,parent,notused,detail
0,2,0,0,CO-ROUTINE order_revenue
1,8,2,216,SCAN order_items
2,10,2,0,USE TEMP B-TREE FOR GROUP BY
3,60,0,50,SEARCH orders USING INDEX idx_orders_status_da...
4,67,0,216,SCAN u
5,74,0,0,BLOOM FILTER ON orv (order_id=?)
6,86,0,47,SEARCH orv USING AUTOMATIC COVERING INDEX (ord...
7,93,0,0,USE TEMP B-TREE FOR GROUP BY
8,132,0,0,USE TEMP B-TREE FOR ORDER BY


In [7]:
time_sql(preagg)

best: 0.9977s | avg: 1.0056s | rows: 5


,channel,revenue
0,organic,20538233.88
1,paid_ads,18821631.27
2,referral,10753845.57
3,email,7838224.43
4,partner,5499758.09


## 4. Индекс для JOIN-ключа

Добавьте индекс на `order_items(order_id)`. Сравните план запроса до и после.

In [8]:
conn.execute('CREATE INDEX IF NOT EXISTS idx_order_items_order ON order_items(order_id)')
explain(preagg)

,id,parent,notused,detail
0,2,0,0,CO-ROUTINE order_revenue
1,9,2,224,SCAN order_items USING INDEX idx_order_items_o...
2,51,0,50,SEARCH orders USING INDEX idx_orders_status_da...
3,58,0,216,SCAN u
4,65,0,0,BLOOM FILTER ON orv (order_id=?)
5,77,0,47,SEARCH orv USING AUTOMATIC COVERING INDEX (ord...
6,84,0,0,USE TEMP B-TREE FOR GROUP BY
7,123,0,0,USE TEMP B-TREE FOR ORDER BY


## 5. EXISTS для проверки наличия действия

Найдите пользователей, у которых был `add_to_cart` в июне 2024. Сначала посмотрите план без индекса, затем добавьте индекс `(event_type, event_time, user_id)`.

In [9]:
exists_query = '''
SELECT u.user_id, u.channel
FROM users u
WHERE EXISTS (
  SELECT 1
  FROM events e
  WHERE e.user_id = u.user_id
    AND e.event_type = 'add_to_cart'
    AND e.event_time >= '2024-06-01'
    AND e.event_time <  '2024-07-01'
)
LIMIT 20
'''
explain(exists_query)

,id,parent,notused,detail
0,5,0,216,SCAN u
1,9,0,0,BLOOM FILTER ON e (user_id=? AND event_type=?)
2,27,0,53,SEARCH e EXISTS USING AUTOMATIC PARTIAL COVERI...


In [10]:
conn.execute('CREATE INDEX IF NOT EXISTS idx_events_type_time_user ON events(event_type, event_time, user_id)')
explain(exists_query)

,id,parent,notused,detail
0,5,0,216,SCAN u
1,7,0,47,SEARCH e EXISTS USING COVERING INDEX idx_event...


## 6. ORDER BY + LIMIT

Найдите 20 последних событий пользователя. Добавьте индекс `(user_id, event_time DESC)` и сравните план.

In [11]:
latest_events = '''
SELECT *
FROM events
WHERE user_id = 42
ORDER BY event_time DESC
LIMIT 20
'''
explain(latest_events)

,id,parent,notused,detail
0,4,0,216,SCAN events
1,21,0,0,USE TEMP B-TREE FOR ORDER BY


In [12]:
conn.execute('CREATE INDEX IF NOT EXISTS idx_events_user_time ON events(user_id, event_time DESC)')
explain(latest_events)

,id,parent,notused,detail
0,5,0,62,SEARCH events USING INDEX idx_events_user_time...
